# Module 6.3 — Multi-Task with Photos

**Module 6: Vision + Deployment | Week 2 | Lecture (Session 6.3)**

*Source: New content (Session 30, 2026-05-18). Built around the empirical record in [`M06-MULTITASK-FINDINGS.md`](../../M06-MULTITASK-FINDINGS.md) — Session 28's pre-investigation that found a +0.017 AUC lift over the flat-concat baseline via shared CNN trunk + auxiliary head. Session 35: SHARED_6 replaces OVERLAP_5 throughout; `is_young` now serves as both tabular input and aux target (dual-channel mechanism); all-dogs training.*

---

In M05-lab4 you built a multi-task model for a *cross-dataset* problem: shared trunk + separate PF and AAC heads, so the trunk's representation could transfer between two shelters. Same architectural pattern shows up today, in a different application.

M06-1's three-model comparison topped out at ~0.695 AUC by concatenating the 6-feature SHARED set and image features into one MLP — *flat concat*. That treats every input dimension equally and asks the optimizer to figure out the structure. We can do better. Today: the same data, the same features, a different architecture — and a meaningful AUC lift that tells you why the architecture matters.

By the end of this lecture you should be able to say, with evidence:

- Adding an auxiliary classification head to a shared CNN trunk lifts adoption AUC by about **+0.032** over flat concat — meaningful and reproducible across seeds.
- The lift is signal transfer, not regularization noise: a *random*-label aux head gives only +0.003 (noise floor); a meaningful aux head (`is_young`) gives ten times that.
- `is_young` does double duty: it's in the SHARED_6 tabular input AND it's the aux target. The tabular input tells the adopt head "this dog is young"; the aux head teaches the trunk "learn to see youth in photos." Two complementary channels.
- The mechanism has **three conditions** in series, and missing any one of them collapses the lift to nothing.
- The aux-loss weight α has a sharp sweet spot around 2.0 — below it the aux signal is too weak to shape the trunk; above it the aux task takes over and adoption performance degrades.
- This is the same pattern as M05-lab4's cross-dataset multi-task, applied to a different problem. 6.5 generalizes back: shared trunk + dataset-specific heads for the Taiwan deployment story.

## Glossary

Most of this carries over from M05-lab4; the new pieces are the loss-weight α and the dual-channel mechanism.

- **Shared trunk** — the layers of the network that *every* prediction passes through. Forced to learn features useful for multiple tasks at once. In M05-lab4 the trunk learned features that were useful for PF and AAC adoption; today it learns features useful for `adopted_in_30_days` and an auxiliary target.
- **Auxiliary head** — a second output head that predicts a side target alongside the main one. The model is graded on both, so the trunk has to encode features useful for both.
- **α (alpha)** — the weight on the auxiliary loss in the combined objective: `loss = adopt_BCE + α · aux_BCE`. α controls how much the aux task gets to shape the trunk's representation. Too low and the trunk ignores the aux signal; too high and the trunk gets hijacked by the aux task.
- **Dual-channel mechanism** — when `is_young` appears as both a SHARED_6 tabular input AND as the aux target. The aux head teaches the trunk "learn to see youth in photos" (photo channel); the tabular input gives the adopt head the ground-truth age flag directly (tabular channel). Two complementary routes to the same information.
- **Three-condition mechanism** — for the aux head to help adoption prediction, three things must all hold: (1) the aux target is *predictable from photos* through the trunk; (2) the aux target *correlates* with the adoption target; (3) α sits in the productive range. We test each condition explicitly below.

## Quick recap — multi-task in M05-lab4

M05-lab4 framed multi-task as a **cross-dataset** tool. The shared trunk consumed 6 features that PetFinder and AAC both had (age, gender, sterilized, mixed-breed, etc.). Each dataset had its own head that consumed `concat(trunk_out, dataset-specific features)` to produce a per-dataset prediction. The lab's headline finding: the **trunk's representation transferred** across shelters (trunk-LR on AAC labels beat the §12 PF-only baseline by +0.066 AUC for dogs) even though the heads themselves did not.

Same architectural pattern today, different application. Replace AAC with an *auxiliary classification target on the same dataset* (`is_young`) and you get a single-dataset boost that lifts adoption AUC by +0.032 over flat concat. The extra lift over earlier experiments comes from the dual-channel mechanism — `is_young` is both a SHARED_6 tabular input and the aux target. 6.5 will generalize the framework back to cross-dataset Taiwan deployment, where the heads change again. The trunk is the load-bearing piece; the heads change to suit the application.

## Setup — load data, multi-photo features, and the auxiliary targets

Same two bundles as 6.1/6.2 — `petfinder-mini` + `m06-photos`. New: alongside the adoption target we derive `is_young` (Age < 6 months) as the auxiliary target. Note that `is_young` is already in the SHARED_6 tabular input — that's intentional. The dual-channel mechanism is the point: `is_young` enters the model through the tabular input (direct demographic signal to the adopt head) *and* through the aux head (teaching the trunk to encode youth from photos).

In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

PETFINDER_URL  = 'CONFIGURE_ME'  # ← your instructor will give you the M05 bundle URL
M06_PHOTOS_URL = 'CONFIGURE_ME'  # ← your instructor will give you the M06 bundle URL
DATA_DIR = '/content/petfinder'
os.makedirs(DATA_DIR, exist_ok=True)

def fetch_bundle(url, marker_file, label):
    if os.path.exists(f'{DATA_DIR}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f'Set {label} above to your instructor URL.')
    import gdown
    zip_path = f'{DATA_DIR}/_bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    if not os.path.exists(zip_path):
        raise RuntimeError(f'{label} download failed — check URL and Drive share settings.')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    if not os.path.exists(f'{DATA_DIR}/{marker_file}'):
        raise RuntimeError(f'{marker_file} not found after extraction — wrong zip?')

fetch_bundle(PETFINDER_URL,  'train.csv',                   'PETFINDER_URL')
fetch_bundle(M06_PHOTOS_URL, 'petfinder_cnn_features.csv', 'M06_PHOTOS_URL')

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Bundles ready in {DATA_DIR}; device={device}')


In [ ]:
df = pd.read_csv(f'{DATA_DIR}/train.csv')
dogs = df[df['Type'] == 1].copy().reset_index(drop=True)

IMAGE = [f'img_{i}' for i in range(768)]
photo_df = pd.read_csv(f'{DATA_DIR}/petfinder_cnn_features.csv').set_index('PetID')

# Left join: include zero-photo dogs (zero-padded CNN).
dogs = dogs.merge(photo_df, left_on='PetID', right_index=True, how='left')
has_photo = dogs[IMAGE[0]].notna().values
dogs[IMAGE] = dogs[IMAGE].fillna(0.0)
dogs = dogs.reset_index(drop=True)

# Main target + aux target.
y_adopt = (dogs['AdoptionSpeed'] <= 2).astype(int).values
y_aux_young = (dogs['Age'] < 6).astype(int).values
y_aux_mixed = ((dogs['Breed1'] == 307) | (dogs['Breed2'] != 0)).astype(int).values

# SHARED_6 tabular — is_young is both here AND the aux target (dual-channel).
X_tab = pd.DataFrame({
    'age_adult':      (dogs['Age'] >= 12).astype(int),
    'is_young':       (dogs['Age'] < 6).astype(int),
    'is_mixed_breed': ((dogs['Breed1'] == 307) | (dogs['Breed2'] != 0)).astype(int),
    'gender_male':    (dogs['Gender'] == 1).astype(int),
    'gender_female':  (dogs['Gender'] == 2).astype(int),
    'sterilized_yes': (dogs['Sterilized'] == 1).astype(int),
}).values.astype(np.float32)
X_img = dogs[IMAGE].values.astype(np.float32)

tr, te = train_test_split(
    np.arange(len(y_adopt)), test_size=0.25, random_state=42, stratify=y_adopt,
)
te_eval = te[has_photo[te]]
y_tr, y_te = y_adopt[tr], y_adopt[te_eval]

def scale(X):
    s = StandardScaler().fit(X[tr])
    return s.transform(X[tr]).astype(np.float32), s.transform(X[te_eval]).astype(np.float32)

X_tab_tr, X_tab_te = scale(X_tab)
X_img_tr, X_img_te = scale(X_img)
X_concat_tr = np.concatenate([X_tab_tr, X_img_tr], axis=1)
X_concat_te = np.concatenate([X_tab_te, X_img_te], axis=1)

print(f'Dogs: {len(dogs):,} total ({has_photo.sum():,} with photos)')
print(f'Train: {len(tr):,} (all dogs)  |  Test: {len(te_eval):,} (photo-having)')
print(f'adopted_in_30_days base rate: {y_adopt.mean():.3f}')
print(f'is_young base rate:           {y_aux_young.mean():.3f}')
print(f'Tabular features: {X_tab.shape[1]} (SHARED_6)')


## The architecture — shared CNN trunk + two heads

M06-1's flat concat treated `[tab_6, img_768]` as one big 774-dim feature vector and dropped it into a single MLP. That works, but the optimizer has to discover the structure from scratch. The multi-task architecture *imposes* structure: the 768 image features go through a shared trunk first; only then are the tabular features concatenated for the adoption prediction; and a second head predicts the auxiliary target from the trunk alone.

```
              multi-photo ConvNeXt-Tiny features (768-dim)
                              │
                              ▼
                   ┌──────────────────────┐
                   │  Shared CNN trunk    │   768 → 64 → 32, ReLU + Dropout(0.3)
                   └────────┬─────────────┘
                            │
                ┌───────────┴────────────┐
                ▼                        ▼
        ┌────────────────┐    ┌──────────────────────────┐
        │  Aux head      │    │  concat(trunk_out, tab_6)│
        │  Linear(32→1)  │    │  → Linear(38 → 16) → ReLU+Dropout
        │  → is_young    │    │  → Linear(16 → 1) → adopt_30
        │  (BCE)         │    │  (BCE)                   │
        └────────────────┘    └──────────────────────────┘

Loss = adopt_BCE + α · aux_BCE
```

Three things worth flagging about this design:

- **The aux head sees the trunk output, not the trunk output + tab.** It only gets photo-derived features. That's how we test whether `is_young` is *predictable from photos*.
- **The adopt head sees `concat(trunk_out, tab_6)`** — the *post-trunk* image representation plus the raw tabular features (including `is_young`). The trunk's job is to refine the image features under both supervisions; the adopt head's job is to combine that refined representation with the tabular columns.
- **`is_young` appears in both paths.** It's in the 6-feature SHARED tabular input (direct demographic signal to the adopt head) AND it's the aux target (teaching the trunk what "young" looks like in photos). This dual-channel mechanism is why SHARED_6 + MultiTask lifts more than OVERLAP_5 + MultiTask did — the tabular channel and the photo channel carry complementary information about the same feature.

In [ ]:
class FlatMLP(nn.Module):
    """M06-1's baseline: one MLP on concat(tab_6, img_768)."""

    def __init__(self, n_in, hidden=(64, 32), dropout=0.3):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MultiTaskMLP(nn.Module):
    """Shared CNN trunk + aux head + concat-with-tab adopt head."""

    def __init__(self, n_img, n_tab, trunk=(64, 32), adopt_head=(16,), dropout=0.3):
        super().__init__()
        layers, prev = [], n_img
        for h in trunk:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.trunk = nn.Sequential(*layers)
        trunk_dim = prev
        self.aux_head = nn.Linear(trunk_dim, 1)
        layers, prev = [], trunk_dim + n_tab
        for h in adopt_head:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.adopt_head = nn.Sequential(*layers)

    def forward(self, x_img, x_tab):
        z = self.trunk(x_img)
        aux_logit = self.aux_head(z).squeeze(-1)
        adopt_logit = self.adopt_head(torch.cat([z, x_tab], dim=1)).squeeze(-1)
        return adopt_logit, aux_logit


In [ ]:
def to_t(arr):
    return torch.tensor(arr, dtype=torch.float32, device=device)


def train_baseline(seed, epochs=150):
    """M06-1's flat-concat baseline. Returns (acc, auc)."""
    torch.manual_seed(seed)
    model = FlatMLP(X_concat_tr.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xtr_t, ytr_t, Xte_t = to_t(X_concat_tr), to_t(y_tr), to_t(X_concat_te)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss_fn(model(Xtr_t), ytr_t).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xte_t)).cpu().numpy()
    return float(((probs > 0.5) == y_te).mean()), float(roc_auc_score(y_te, probs))


def train_multitask(y_aux, seed, alpha, epochs=150):
    """Multi-task: shared trunk + aux head + adopt head. Returns (acc, auc) on adopt."""
    torch.manual_seed(seed)
    model = MultiTaskMLP(n_img=X_img_tr.shape[1], n_tab=X_tab_tr.shape[1]).to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCEWithLogitsLoss()
    Xi_tr, Xt_tr = to_t(X_img_tr), to_t(X_tab_tr)
    ya_tr, yad_tr = to_t(y_aux[tr]), to_t(y_tr)
    Xi_te, Xt_te = to_t(X_img_te), to_t(X_tab_te)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        adopt_logit, aux_logit = model(Xi_tr, Xt_tr)
        loss = loss_fn(adopt_logit, yad_tr) + alpha * loss_fn(aux_logit, ya_tr)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        adopt_logit, _ = model(Xi_te, Xt_te)
        probs = torch.sigmoid(adopt_logit).cpu().numpy()
    return float(((probs > 0.5) == y_te).mean()), float(roc_auc_score(y_te, probs))

In [ ]:
# Baseline: M06-1's flat concat, 3-seed mean.
SEEDS = [42, 1, 2]

base_accs, base_aucs = [], []
for s in SEEDS:
    a, u = train_baseline(s)
    base_accs.append(a); base_aucs.append(u)
BASE_ACC, BASE_AUC = float(np.mean(base_accs)), float(np.mean(base_aucs))
print(f'Flat-concat baseline (3-seed mean):  acc {BASE_ACC:.4f}  AUC {BASE_AUC:.4f}')
print(f'  per-seed AUCs: {[round(u, 4) for u in base_aucs]}')

In [ ]:
# Multi-task with is_young aux at α=2.0, 3-seed mean.
mt_accs, mt_aucs = [], []
for s in SEEDS:
    a, u = train_multitask(y_aux_young, seed=s, alpha=2.0)
    mt_accs.append(a); mt_aucs.append(u)
MT_ACC, MT_AUC = float(np.mean(mt_accs)), float(np.mean(mt_aucs))
print(f"Multi-task is_young α=2.0 (3-seed):  acc {MT_ACC:.4f}  AUC {MT_AUC:.4f}")
print(f"  per-seed AUCs: {[round(u, 4) for u in mt_aucs]}")
print(f"  Δ vs baseline:                       Δacc {MT_ACC - BASE_ACC:+.4f}  ΔAUC {MT_AUC - BASE_AUC:+.4f}")

### Reading the result — the three-condition mechanism

On 3 seeds your multi-task lift should land near +0.032 AUC (within ±0.008). That's nearly double the +0.017 you'd get if `is_young` were *only* an aux target (as it was with OVERLAP_5). The extra lift comes from the dual-channel mechanism: `is_young` enters the model through both the tabular input and the aux head, and the two channels are complementary rather than redundant.

The lift comes from a chain of three conditions. **Miss any one and the lift collapses to noise.**

1. **Photo-predictability.** The CNN trunk must be able to predict the aux target from photos alone. Young dogs *look* young — body proportions, fur texture, eye shape. A photo-only model gets ~0.94 AUC predicting `is_young`. Sentiment, by contrast, is invisible to a CNN: photo-only AUC for sentiment > 0.3 is 0.558 — barely above random. If the trunk can't predict the aux target, the aux gradient has nothing to teach it.
2. **Adoption-correlation.** The aux target must actually correlate with adoption. `is_young` predicts adoption with +0.129 rate-gap (M05-3's feature ranking); `is_mixed_breed` predicts +0.018. If the aux target had nothing to do with adoption, the trunk's improved aux representation would help adoption by zero.
3. **α budget.** The aux loss weight α must sit in a productive range. Too low (~0.5) and the aux signal is too weak to influence the trunk. Too high (~8.0) and the aux task hijacks the trunk's representation, degrading adoption. The peak is at α=2.0 and is fairly sharp.

Restated: **the aux loss teaches the trunk to encode photo features that distinguish young vs old.** Those same features turn out to be useful for adoption prediction (because age affects adoption rates). Meanwhile, the tabular `is_young` flag gives the adopt head the *ground truth* age directly — the two channels reinforce each other. The aux head shapes the trunk representation from the photo side; the tabular input gives the adopt head the demographic fact from the data side.

In [ ]:
# Condition 3 in action — α sweep on is_young. Peak should land at α=2;
# higher α collapses adoption performance as the aux task takes over.
alphas = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0]
alpha_rows = []
for alpha in alphas:
    accs, aucs = [], []
    for s in SEEDS:
        a, u = train_multitask(y_aux_young, seed=s, alpha=alpha)
        accs.append(a); aucs.append(u)
    alpha_rows.append({
        'α':       alpha,
        'acc':     float(np.mean(accs)),
        'AUC':     float(np.mean(aucs)),
        'ΔAUC':    float(np.mean(aucs)) - BASE_AUC,
    })
print(pd.DataFrame(alpha_rows).to_string(index=False, float_format='%.4f'))

In [ ]:
# Condition 1+2 in action — what happens with RANDOM aux labels?
# If the lift were just regularization (more parameters, more loss terms),
# random labels would still help. If the lift is signal transfer, random
# labels should give essentially nothing.
rng = np.random.default_rng(0)
y_aux_random = rng.integers(0, 2, size=len(y_adopt)).astype(int)

rand_accs, rand_aucs = [], []
for s in SEEDS:
    a, u = train_multitask(y_aux_random, seed=s, alpha=2.0)
    rand_accs.append(a); rand_aucs.append(u)
RAND_AUC = float(np.mean(rand_aucs))
print(f'Random aux labels α=2.0 (3-seed):    AUC {RAND_AUC:.4f}  ΔAUC {RAND_AUC - BASE_AUC:+.4f}')
print(f'is_young aux α=2.0 (recap):          AUC {MT_AUC:.4f}  ΔAUC {MT_AUC - BASE_AUC:+.4f}')
print()
print('Random gives the noise floor. is_young clears it by ~5x — the lift is signal, not regularization.')

## How close is the deployable model to the ceiling?

SHARED_6 has 6 features — all available in Taiwan COA data, so the model is deployable as-is. What if you used *every* feature PetFinder has (44 features including description length, fee, color codes, etc.)? The answer: FULL_PLUS + CNN + MultiTask reaches AUC 0.733. The gap from SHARED_6 (0.727) to the ceiling is **less than 0.01 AUC**. Thirty-eight extra features buy almost nothing on top of the multi-task architecture with good demographics.

That's the practical message: SHARED_6 + CNN + MultiTask is both the deployable model and very close to the best model possible on this data. The deep dive `M06-deep_sentiment_subset.ipynb` explores richer tabular configurations and metadata-as-aux-targets if you want to probe the ceiling further.

## What invisible signals look like — forecast to M07

We've shown that `is_young` works because a photo-only model can predict it. The opposite case is just as informative: **`sentiment > 0.3` does almost nothing** as an aux target (+0.009 AUC over baseline in earlier experiments), and the reason is that a photo-only model gets only 0.558 AUC trying to predict sentiment from images. Sentiment lives in the listing's *description*, not the photo; the trunk can't learn to encode something it can't see.

That's the boundary of M06's photo pipeline. To get a sentiment lift you need a model that can actually read the text — a text encoder (transformer-based, like BERT or DistilBERT) that produces a feature vector for the description. **That's M07.** The architectural picture for M07 isn't far from today's lecture: a different encoder (text instead of CNN), the same shared-trunk-plus-heads pattern, and now the trunk learns features useful across modalities.

Bottom line: photo-derived features plus the dual-channel mechanism lift adoption AUC to 0.727. Description-derived features should lift further, but only through a text-aware model. The CNN trunk is the wrong place for sentiment; the right place is a transformer. M07 picks this up.

---

## Summary

**Key Moment 1 of M06: SHARED_6 + CNN + MultiTask = AUC 0.727.** One feature set, one architecture, the best deployable model. This is the number the Taiwan deployment in 6.5 builds on.

- Multi-task with a shared CNN trunk + auxiliary head lifts adoption AUC by about **+0.032** over M06-1's flat-concat baseline (SHARED_6 + CNN FlatMLP at ~0.695). Reproducible across seeds.
- The lift is **signal transfer, not regularization** — random-label aux gives only +0.003 (noise floor).
- **`is_young` does double duty** — it's in the SHARED_6 tabular input AND it's the aux target. The aux head shapes the trunk representation from photos; the tabular input gives the adopt head the demographic fact directly. Two complementary channels.
- The three-condition mechanism: photo-predictability + adoption-correlation + α budget. All three must hold or the lift collapses.
- α=2.0 is the sweet spot; α=8.0 starts collapsing adoption performance.
- The deployable gap (SHARED_6 at 0.727 vs FULL_PLUS at 0.733) is **less than 0.01 AUC**. Thirty-eight extra features buy almost nothing on top of the multi-task architecture.
- Sentiment can't work as a photo-side aux because the CNN trunk can't predict sentiment from images. That's the M07 handoff: text encoder, same multi-task pattern.

### Where M06 goes from here

- **6.4 — Lab:** You build this architecture end-to-end — reproduce the baseline, hit the +0.032 with is_young at α=2.0, sweep α to find the peak, swap aux targets including a random control. Stretch: remove `is_young` from both channels to measure its full contribution.
- **6.5–6.6:** The same shared-trunk pattern returns in its M05-lab4 form — shared trunk + dataset-specific heads — applied to the cross-shelter Taiwan COA deployment story. Same SHARED_6 features, same architecture.
- **6.7 onwards:** Responsible vision-ML, pre-capstone, and the Taiwan adoption brief itself.

---

**Next session (Lab 6.4):** Build the multi-task architecture, sweep α, swap aux targets. Same data, same architecture, your hands on the knobs.